# Chapter 12 — Designing and Deploying Command Line Programs

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 12](https://automatetheboringstuff.com/3e/chapter12.html)

**Setup note:** sys.argv/argparse only make sense for scripts run from a terminal — in this notebook they're shown as code you'd put in a .py file.

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 12.1 sys.argv: raw command line arguments

When you run `python myscript.py foo bar`, Python stores `['myscript.py', 'foo', 'bar']` in `sys.argv`. It's simple but you have to parse types, flags, and validation yourself.


In [1]:
# Save as greet.py and run: python greet.py Alice
script = '''
import sys
if len(sys.argv) < 2:
    print("Usage: python greet.py <name>")
else:
    print("Hello, " + sys.argv[1])
'''
print(script)



import sys
if len(sys.argv) < 2:
    print("Usage: python greet.py <name>")
else:
    print("Hello, " + sys.argv[1])



## 12.2 argparse: proper CLI interfaces

`argparse` (standard library) builds a real command line interface: named/positional arguments, types, defaults, `--help` text, and error messages — all for free. This is what real-world CLI tools use instead of hand-parsing `sys.argv`.


In [2]:
# Save as greet2.py and run: python greet2.py Alice --shout
script = '''
import argparse

parser = argparse.ArgumentParser(description="Greet someone.")
parser.add_argument("name", help="the name to greet")
parser.add_argument("--shout", action="store_true", help="use all caps")
args = parser.parse_args()

message = f"Hello, {args.name}!"
print(message.upper() if args.shout else message)
'''
print(script)



import argparse

parser = argparse.ArgumentParser(description="Greet someone.")
parser.add_argument("name", help="the name to greet")
parser.add_argument("--shout", action="store_true", help="use all caps")
args = parser.parse_args()

message = f"Hello, {args.name}!"
print(message.upper() if args.shout else message)



## 12.3 Reading piped input from stdin

CLI tools often need to work with pipes, e.g. `cat file.txt | python myscript.py`. `sys.stdin` behaves like a file object you can iterate over line by line.


In [3]:
script = '''
import sys
for line in sys.stdin:
    print("Got:", line.strip())
'''
print(script)



import sys
for line in sys.stdin:
    print("Got:", line.strip())



## 12.4 Logging instead of print() for CLI tools

For a real command line tool, route status/debug messages through `logging` (to stderr) so users can silence them with a flag, while `print()` stays reserved for the actual program output (which may be piped into another program).


In [4]:
import logging, sys
logging.basicConfig(level=logging.INFO, stream=sys.stderr, format='%(levelname)s: %(message)s')

logging.info('Starting processing...')
print('actual output line 1')
logging.info('Done.')


INFO: Starting processing...


INFO: Done.


actual output line 1


## 12.5 Compiling Python programs with PyInstaller

Sharing a script with someone who doesn't have Python installed means either asking them to install Python first, or **compiling** the script into a standalone executable with the third-party `PyInstaller` tool: `pyinstaller --onefile myscript.py` produces a single binary (in `dist/`) that runs on that same OS with no Python installation required — it bundles the interpreter and every imported library together. This is genuinely a real compilation step (not just an interpreter wrapper), which is why the book highlights it as the way to *deploy* a finished script rather than just run it from source.

In [5]:
import subprocess, pathlib, shutil, sys

work = pathlib.Path('pyinstaller_demo')
work.mkdir(exist_ok=True)
script = work / 'hello_cli.py'
script.write_text("print('Hello from a COMPILED executable, no Python install needed to run this file!')\n")

result = subprocess.run(
    ['pyinstaller', '--onefile', '--distpath', str(work / 'dist'),
     '--workpath', str(work / 'build'), '--specpath', str(work), str(script)],
    capture_output=True, text=True
)
print('PyInstaller exit code:', result.returncode)

exe = work / 'dist' / 'hello_cli'
print('Executable created:', exe.exists())
if exe.exists():
    run = subprocess.run([str(exe)], capture_output=True, text=True)
    print('Running the compiled binary ->', run.stdout.strip())

shutil.rmtree(work)   # clean up the demo build artifacts


PyInstaller exit code: 0
Executable created: True
Running the compiled binary -> Hello from a COMPILED executable, no Python install needed to run this file!


## 12.6 Isolating dependencies with venv

Before compiling or sharing a script, you generally want to know *exactly* which third-party packages it needs — which is hard to tell if everything is installed globally. The built-in `venv` module creates an isolated Python environment with its own separate package set: `python3 -m venv myenv` creates it, then activating it (`source myenv/bin/activate` on Linux/macOS, `myenv\Scripts\activate` on Windows) makes `pip install` affect only that environment, not your whole system. This is the standard way to keep one project's dependencies from clashing with another's.

In [6]:
import subprocess, pathlib, shutil, sys

venv_dir = pathlib.Path('demo_venv')
subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)

venv_python = venv_dir / 'bin' / 'python3'
result = subprocess.run([str(venv_python), '--version'], capture_output=True, text=True)
print('Python inside the isolated venv:', result.stdout.strip() or result.stderr.strip())

# Prove it's isolated: a package installed system-wide isn't automatically visible inside it
check = subprocess.run([str(venv_python), '-c', 'import humre; print("humre importable inside venv")'],
                        capture_output=True, text=True)
print('Trying to import a globally-installed package from inside the venv:')
print(' ', (check.stdout + check.stderr).strip().splitlines()[-1])

shutil.rmtree(venv_dir)


Python inside the isolated venv: Python 3.10.12
Trying to import a globally-installed package from inside the venv:
  ModuleNotFoundError: No module named 'humre'


## 12.7 Colored terminal output with Bext

Plain `print()` is monochrome. The third-party `Bext` module wraps the ANSI escape codes that terminals understand to add color, move the cursor, resize the window, or clear the screen — useful for simple text-based dashboards or games run from the command line. `bext.fg('red')` switches the foreground color for everything printed afterward; `bext.fg('reset')` (or restarting the terminal) returns to normal.

In [7]:
import bext

bext.fg('green')
print('This prints in green (in a real terminal that supports ANSI colors).')
bext.fg('red')
print('This prints in red.')
bext.fg('reset')
print('Back to the default color.')

print()
print('Note: a Jupyter notebook cell renders plain text, so you will only SEE the color')
print('change when this exact code runs inside a real terminal window (e.g. via Cursor\'s')
print('integrated terminal) -- the ANSI codes are still genuinely being emitted here.')


This prints in green (in a real terminal that supports ANSI colors).

This prints in red.

Back to the default color.

Note: a Jupyter notebook cell renders plain text, so you will only SEE the color

change when this exact code runs inside a real terminal window (e.g. via Cursor's

integrated terminal) -- the ANSI codes are still genuinely being emitted here.

## 12.8 Simple GUI dialogs with PyMsgBox

Sometimes a CLI tool needs one quick GUI popup instead of a full interface — `PyMsgBox` provides exactly that: `pymsgbox.alert('message')` shows an OK-only popup, `confirm('question')` shows OK/Cancel and returns which was clicked, `prompt('question')` shows a popup with a text field, and `password('question')` does the same but masks the input. All four block program execution until the user responds, then return the user's choice/input as a string.

**Environment note:** these dialogs need Tkinter and a real display (a windowing system) to draw a window — this headless sandbox has neither, so the call raises immediately. This is expected and identical in spirit to Chapter 23's mouse/keyboard code: the API is real, but it needs a real desktop to run.

In [8]:
import pymsgbox

try:
    response = pymsgbox.confirm('Deploy to production?', buttons=['Yes', 'No'])
    print('User clicked:', response)
except Exception as e:
    print(f'{type(e).__name__} in this sandbox (expected, no display/Tkinter available here):', e)
    print('On your own desktop, this would pop up a real OK/Cancel-style dialog box.')


AssertionError in this sandbox (expected, no display/Tkinter available here):

Tkinter is required for pymsgbox

On your own desktop, this would pop up a real OK/Cancel-style dialog box.